In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [2]:
day9_file_path = Path(
    r"D:\Data Analytics Project\EduPro_Predictive_Modeling\data\model_data\EduPro_Day9_Train_Test_Splits.xlsx"
)

print("Day 9 input file:")
print(day9_file_path)

print("File exists:", day9_file_path.exists())

Day 9 input file:
D:\Data Analytics Project\EduPro_Predictive_Modeling\data\model_data\EduPro_Day9_Train_Test_Splits.xlsx
File exists: True


In [4]:
train_data = pd.read_excel(
    day9_file_path,
    sheet_name="Train_Data"
)

test_data = pd.read_excel(
    day9_file_path,
    sheet_name="Test_Data"
)

X_train = pd.read_excel(
    day9_file_path,
    sheet_name="X_Train"
)

X_test = pd.read_excel(
    day9_file_path,
    sheet_name="X_Test"
)

y_train = pd.read_excel(
    day9_file_path,
    sheet_name="Y_Train"
)

y_test = pd.read_excel(
    day9_file_path,
    sheet_name="Y_Test"
)

print("Day 9 datasets loaded successfully.")

print("\nTrain_Data:", train_data.shape)
print("Test_Data:", test_data.shape)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

Day 9 datasets loaded successfully.

Train_Data: (48, 15)
Test_Data: (12, 15)
X_train: (48, 9)
X_test: (12, 9)
y_train: (48, 2)
y_test: (12, 2)


## Validating Day 9

In [6]:
print("========== DAY 10 INPUT VALIDATION ==========")

print("\nTraining rows:")
print(len(X_train))

print("\nTesting rows:")
print(len(X_test))

print("\nTraining features:")
print(X_train.shape)

print("\nTesting features:")
print(X_test.shape)

print("\nTraining targets:")
print(y_train.shape)

print("\nTesting targets:")
print(y_test.shape)

print("\nTraining missing values:")
print(X_train.isnull().sum().sum() + y_train.isnull().sum().sum())

print("\nTesting missing values:")
print(X_test.isnull().sum().sum() + y_test.isnull().sum().sum())

========== DAY 10 INPUT VALIDATION ==========

Training rows:
48

Testing rows:
12

Training features:
(48, 9)

Testing features:
(12, 9)

Training targets:
(48, 2)

Testing targets:
(12, 2)

Training missing values:
0

Testing missing values:
0


## Define targets & features

In [7]:
target_columns = [
    "EnrollmentCount",
    "CourseRevenue"
]

feature_columns = [
    "CourseCategory",
    "CourseType",
    "CourseLevel",
    "CoursePrice",
    "CourseDuration",
    "CourseRating",
    "TeacherRating",
    "YearsOfExperience",
    "Expertise"
]

print("Target columns:")
print(target_columns)

print("\nFeature columns:")
print(feature_columns)

print("\nNumber of features:")
print(len(feature_columns))

Target columns:
['EnrollmentCount', 'CourseRevenue']

Feature columns:
['CourseCategory', 'CourseType', 'CourseLevel', 'CoursePrice', 'CourseDuration', 'CourseRating', 'TeacherRating', 'YearsOfExperience', 'Expertise']

Number of features:
9


## Identify features types

In [8]:
categorical_features = [
    "CourseCategory",
    "CourseType",
    "CourseLevel",
    "Expertise"
]

numerical_features = [
    "CoursePrice",
    "CourseDuration",
    "CourseRating",
    "TeacherRating",
    "YearsOfExperience"
]

print("Categorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numerical_features)

Categorical features:
['CourseCategory', 'CourseType', 'CourseLevel', 'Expertise']

Numerical features:
['CoursePrice', 'CourseDuration', 'CourseRating', 'TeacherRating', 'YearsOfExperience']


## Create Mean Baselines

In [10]:
mean_baseline_predictions = {}

for target in target_columns:
    
    train_mean = y_train[target].mean()
    
    mean_baseline_predictions[target] = np.full(
        shape=len(y_test),
        fill_value=train_mean
    )
    
    print(f"{target} training mean: {train_mean:.4f}")

EnrollmentCount training mean: 166.1667
CourseRevenue training mean: 11955.5406


## Create preprosessing pipeline

In [11]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numerical_features
        )
    ]
)

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


## Create Linear Regression model

In [12]:
linear_regression_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)

print("Linear Regression pipeline created successfully.")

Linear Regression pipeline created successfully.


## Train Linear Regression for EnrollmentCount

In [13]:
linear_regression_enrollment = linear_regression_pipeline

linear_regression_enrollment.fit(
    X_train,
    y_train["EnrollmentCount"]
)

enrollment_predictions = linear_regression_enrollment.predict(
    X_test
)

print("Linear Regression trained for EnrollmentCount.")

print("\nFirst 5 predictions:")
print(enrollment_predictions[:5])

Linear Regression trained for EnrollmentCount.

First 5 predictions:
[162.2599605  173.32232955 121.22478986 147.50224944 170.06496777]


## Train Linear Regression for CourseRevenue

In [14]:
linear_regression_revenue = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)

linear_regression_revenue.fit(
    X_train,
    y_train["CourseRevenue"]
)

revenue_predictions = linear_regression_revenue.predict(
    X_test
)

print("Linear Regression trained for CourseRevenue.")

print("\nFirst 5 predictions:")
print(revenue_predictions[:5])

Linear Regression trained for CourseRevenue.

First 5 predictions:
[80018.23592086   453.45948324 62955.66192014 -3495.88146106
  9883.68759419]


## Calculate evalution metrics

In [15]:
results = []

for target in target_columns:
    
    actual = y_test[target]
    
    # Mean Baseline
    mean_pred = mean_baseline_predictions[target]
    
    mean_mae = mean_absolute_error(actual, mean_pred)
    mean_rmse = np.sqrt(mean_squared_error(actual, mean_pred))
    mean_r2 = r2_score(actual, mean_pred)
    
    results.append({
        "Target": target,
        "Model": "Mean Baseline",
        "MAE": mean_mae,
        "RMSE": mean_rmse,
        "R2": mean_r2
    })


# EnrollmentCount Linear Regression
enrollment_mae = mean_absolute_error(
    y_test["EnrollmentCount"],
    enrollment_predictions
)

enrollment_rmse = np.sqrt(
    mean_squared_error(
        y_test["EnrollmentCount"],
        enrollment_predictions
    )
)

enrollment_r2 = r2_score(
    y_test["EnrollmentCount"],
    enrollment_predictions
)

results.append({
    "Target": "EnrollmentCount",
    "Model": "Linear Regression",
    "MAE": enrollment_mae,
    "RMSE": enrollment_rmse,
    "R2": enrollment_r2
})


# CourseRevenue Linear Regression
revenue_mae = mean_absolute_error(
    y_test["CourseRevenue"],
    revenue_predictions
)

revenue_rmse = np.sqrt(
    mean_squared_error(
        y_test["CourseRevenue"],
        revenue_predictions
    )
)

revenue_r2 = r2_score(
    y_test["CourseRevenue"],
    revenue_predictions
)

results.append({
    "Target": "CourseRevenue",
    "Model": "Linear Regression",
    "MAE": revenue_mae,
    "RMSE": revenue_rmse,
    "R2": revenue_r2
})


results_df = pd.DataFrame(results)

results_df

,Target,Model,MAE,RMSE,R2
0,EnrollmentCount,Mean Baseline,10.444444,11.774502,-0.047209
1,CourseRevenue,Mean Baseline,28413.556771,36217.344701,-0.248807
2,EnrollmentCount,Linear Regression,14.203758,19.812472,-1.965007
3,CourseRevenue,Linear Regression,2725.992334,3542.471175,0.988053


## Display formatted results

In [16]:
formatted_results = results_df.copy()

formatted_results["MAE"] = formatted_results["MAE"].round(3)
formatted_results["RMSE"] = formatted_results["RMSE"].round(3)
formatted_results["R2"] = formatted_results["R2"].round(4)

formatted_results

,Target,Model,MAE,RMSE,R2
0,EnrollmentCount,Mean Baseline,10.444,11.775,-0.0472
1,CourseRevenue,Mean Baseline,28413.557,36217.345,-0.2488
2,EnrollmentCount,Linear Regression,14.204,19.812,-1.9650
3,CourseRevenue,Linear Regression,2725.992,3542.471,0.9881


## Compare models automatically

In [17]:
print("========== BASELINE MODEL COMPARISON ==========")

for target in target_columns:
    
    target_results = results_df[
        results_df["Target"] == target
    ].copy()
    
    target_results = target_results.sort_values(
        by="MAE"
    )
    
    print(f"\nTarget: {target}")
    print(target_results)

========== BASELINE MODEL COMPARISON ==========

Target: EnrollmentCount
            Target              Model        MAE       RMSE        R2
0  EnrollmentCount      Mean Baseline  10.444444  11.774502 -0.047209
2  EnrollmentCount  Linear Regression  14.203758  19.812472 -1.965007

Target: CourseRevenue
          Target              Model           MAE          RMSE        R2
3  CourseRevenue  Linear Regression   2725.992334   3542.471175  0.988053
1  CourseRevenue      Mean Baseline  28413.556771  36217.344701 -0.248807


## Determine whether Linear Regression beats Mean Baseline

In [18]:
comparison = []

for target in target_columns:
    
    mean_row = results_df[
        (results_df["Target"] == target) &
        (results_df["Model"] == "Mean Baseline")
    ].iloc[0]
    
    lr_row = results_df[
        (results_df["Target"] == target) &
        (results_df["Model"] == "Linear Regression")
    ].iloc[0]
    
    comparison.append({
        "Target": target,
        "Mean_Baseline_MAE": mean_row["MAE"],
        "Linear_Regression_MAE": lr_row["MAE"],
        "MAE_Improvement": mean_row["MAE"] - lr_row["MAE"],
        "Mean_Baseline_RMSE": mean_row["RMSE"],
        "Linear_Regression_RMSE": lr_row["RMSE"],
        "RMSE_Improvement": mean_row["RMSE"] - lr_row["RMSE"],
        "Mean_Baseline_R2": mean_row["R2"],
        "Linear_Regression_R2": lr_row["R2"]
    })

comparison_df = pd.DataFrame(comparison)

comparison_df

,Target,Mean_Baseline_MAE,Linear_Regression_MAE,MAE_Improvement,Mean_Baseline_RMSE,Linear_Regression_RMSE,RMSE_Improvement,Mean_Baseline_R2,Linear_Regression_R2
0,EnrollmentCount,10.444444,14.203758,-3.759314,11.774502,19.812472,-8.037970,-0.047209,-1.965007
1,CourseRevenue,28413.556771,2725.992334,25687.564437,36217.344701,3542.471175,32674.873526,-0.248807,0.988053


## Create prediction table

In [19]:
prediction_comparison = test_data[
    ["CourseID", "CourseName"]
].copy()

prediction_comparison["Actual_EnrollmentCount"] = (
    y_test["EnrollmentCount"].values
)

prediction_comparison["Mean_Baseline_EnrollmentCount"] = (
    mean_baseline_predictions["EnrollmentCount"]
)

prediction_comparison["Linear_Regression_EnrollmentCount"] = (
    enrollment_predictions
)

prediction_comparison["Actual_CourseRevenue"] = (
    y_test["CourseRevenue"].values
)

prediction_comparison["Mean_Baseline_CourseRevenue"] = (
    mean_baseline_predictions["CourseRevenue"]
)

prediction_comparison["Linear_Regression_CourseRevenue"] = (
    revenue_predictions
)

prediction_comparison

,CourseID,CourseName,Actual_EnrollmentCount,Mean_Baseline_EnrollmentCount,Linear_Regression_EnrollmentCount,Actual_CourseRevenue,Mean_Baseline_CourseRevenue,Linear_Regression_CourseRevenue
0,CR00050,Computer Vision,174,166.166667,162.259960,85416.60,11955.540625,80018.235921
1,CR00036,Agile Project Management,177,166.166667,173.322330,0.00,11955.540625,453.459483
2,CR00049,Deep Learning,163,166.166667,121.224790,67728.13,11955.540625,62955.661920
3,CR00002,Java Programming,149,166.166667,147.502249,0.00,11955.540625,-3495.881461
4,CR00055,Full-Stack Web Development,178,166.166667,170.064968,10202.96,11955.540625,9883.687594
5,CR00034,Data Encryption,150,166.166667,148.912965,59139.00,11955.540625,66994.776669
6,CR00035,Cyber Threat Intelligence,179,166.166667,172.954450,0.00,11955.540625,2678.662070
7,CR00045,Financial Modeling,172,166.166667,177.761644,0.00,11955.540625,972.327454
8,CR00020,Email Marketing,155,166.166667,152.616633,0.00,11955.540625,-279.768222
9,CR00048,AI Ethics,186,166.166667,151.064392,0.00,11955.540625,-1667.015037


## Round prediction results

In [20]:
prediction_comparison = prediction_comparison.round({
    "Actual_EnrollmentCount": 2,
    "Mean_Baseline_EnrollmentCount": 2,
    "Linear_Regression_EnrollmentCount": 2,
    "Actual_CourseRevenue": 2,
    "Mean_Baseline_CourseRevenue": 2,
    "Linear_Regression_CourseRevenue": 2
})

prediction_comparison

,CourseID,CourseName,Actual_EnrollmentCount,Mean_Baseline_EnrollmentCount,Linear_Regression_EnrollmentCount,Actual_CourseRevenue,Mean_Baseline_CourseRevenue,Linear_Regression_CourseRevenue
0,CR00050,Computer Vision,174,166.17,162.26,85416.60,11955.54,80018.24
1,CR00036,Agile Project Management,177,166.17,173.32,0.00,11955.54,453.46
2,CR00049,Deep Learning,163,166.17,121.22,67728.13,11955.54,62955.66
3,CR00002,Java Programming,149,166.17,147.50,0.00,11955.54,-3495.88
4,CR00055,Full-Stack Web Development,178,166.17,170.06,10202.96,11955.54,9883.69
5,CR00034,Data Encryption,150,166.17,148.91,59139.00,11955.54,66994.78
6,CR00035,Cyber Threat Intelligence,179,166.17,172.95,0.00,11955.54,2678.66
7,CR00045,Financial Modeling,172,166.17,177.76,0.00,11955.54,972.33
8,CR00020,Email Marketing,155,166.17,152.62,0.00,11955.54,-279.77
9,CR00048,AI Ethics,186,166.17,151.06,0.00,11955.54,-1667.02


## Define Day 10 output path 

In [21]:
model_folder = Path(
    r"D:\Data Analytics Project\EduPro_Predictive_Modeling\data\model_data"
)

model_folder.mkdir(
    parents=True,
    exist_ok=True
)

day10_file_path = (
    model_folder /
    "EduPro_Day10_Baseline_Models.xlsx"
)

print("Day 10 output file:")
print(day10_file_path)

Day 10 output file:
D:\Data Analytics Project\EduPro_Predictive_Modeling\data\model_data\EduPro_Day10_Baseline_Models.xlsx


## Saving Day 10 output

In [22]:
with pd.ExcelWriter(
    day10_file_path,
    engine="openpyxl"
) as writer:
    
    results_df.to_excel(
        writer,
        sheet_name="Baseline_Metrics",
        index=False
    )
    
    comparison_df.to_excel(
        writer,
        sheet_name="Model_Comparison",
        index=False
    )
    
    prediction_comparison.to_excel(
        writer,
        sheet_name="Predictions",
        index=False
    )

print("✅ Day 10 baseline model results saved successfully:")
print(day10_file_path)

✅ Day 10 baseline model results saved successfully:
D:\Data Analytics Project\EduPro_Predictive_Modeling\data\model_data\EduPro_Day10_Baseline_Models.xlsx


## Reload and validate output

In [23]:
check_metrics = pd.read_excel(
    day10_file_path,
    sheet_name="Baseline_Metrics"
)

check_comparison = pd.read_excel(
    day10_file_path,
    sheet_name="Model_Comparison"
)

check_predictions = pd.read_excel(
    day10_file_path,
    sheet_name="Predictions"
)

print("========== DAY 10 OUTPUT VALIDATION ==========")

print("\nBaseline metrics:")
print(check_metrics.shape)

print("\nModel comparison:")
print(check_comparison.shape)

print("\nPredictions:")
print(check_predictions.shape)

print("\nOutput file exists:")
print(day10_file_path.exists())

========== DAY 10 OUTPUT VALIDATION ==========

Baseline metrics:
(4, 5)

Model comparison:
(2, 9)

Predictions:
(12, 8)

Output file exists:
True


## Final Day 10 validation

In [24]:
print("==============================================")
print("DAY 10 BASELINE MODEL VALIDATION")
print("==============================================")

print("\nInput file:")
print(day9_file_path)

print("\nOutput file:")
print(day10_file_path)

print("\nTraining observations:")
print(len(X_train))

print("\nTesting observations:")
print(len(X_test))

print("\nNumber of features:")
print(len(feature_columns))

print("\nTargets:")
print(target_columns)

print("\nModels evaluated:")
print([
    "Mean Baseline",
    "Linear Regression"
])

print("\nMetrics:")
print([
    "MAE",
    "RMSE",
    "R2"
])

print("\nOutput file exists:")
print(day10_file_path.exists())

print("\n==============================================")
print("DAY 10 COMPLETED")
print("==============================================")

DAY 10 BASELINE MODEL VALIDATION

Input file:
D:\Data Analytics Project\EduPro_Predictive_Modeling\data\model_data\EduPro_Day9_Train_Test_Splits.xlsx

Output file:
D:\Data Analytics Project\EduPro_Predictive_Modeling\data\model_data\EduPro_Day10_Baseline_Models.xlsx

Training observations:
48

Testing observations:
12

Number of features:
9

Targets:
['EnrollmentCount', 'CourseRevenue']

Models evaluated:
['Mean Baseline', 'Linear Regression']

Metrics:
['MAE', 'RMSE', 'R2']

Output file exists:
True

DAY 10 COMPLETED
